# InsuML benchmark — Ludwig backend

Runs Ludwig's AutoML engine (`ludwig.automl.auto_train`) across the 14 OpenML
benchmark tasks, using OpenML's official cross-validation folds (fold i
with seed i, so 10 runs per task; results are averaged over them, see
`common_utils.py`).

Requires: `requirements.txt` (Ludwig 0.17.9).
Shares its experimental protocol (task list, folds, seeds) with
`sklearn_experiment.ipynb` via `common_utils.py`, so results from both
notebooks are directly comparable.

In [ ]:
import os
import sys

# common_utils.py lives in the parent folder (shared by both notebooks)
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
import json
import time

from common_utils import disable_unsupported_gpu, cleanup_ludwig_artifacts

# Must run before importing ludwig/torch (falls back to CPU on old GPUs)
disable_unsupported_gpu()

from ludwig.automl import auto_train

from common_utils import (
    TASK_IDS, TEST_MODE, RUN_MODE_NAME, TIME_LIMIT_S,
    get_fold_repeat_seed_plan,
    save_environment_info,
    load_completed_runs,
    RESUME,
    start_fresh_if_requested,
    DATASET_RETRIES,
    archive_task_results,
    extra_classification_metrics,
    download_openml_task_split,
    safe_metric_value,
    save_incremental_results,
    save_summary,
)

import openml

In [ ]:
# Time limit per Ludwig run (seconds)
LUDWIG_TIME_LIMIT = TIME_LIMIT_S  # set in common_utils.py

## Ludwig experiment functions

In [ ]:
def load_best_model_local(results):
    """Loads the best trial's model on the local backend.

    Ludwig's `results.best_model` calls LudwigModel.load() without a backend,
    so it auto-detects Ray (already running for hyperopt) and fails with
    "RayTrainerV2 has no attribute create_checkpoint_handle". This does the
    same reload but passes backend="local" explicitly.
    """
    from ludwig.api import LudwigModel
    from ludwig.globals import MODEL_FILE_NAME

    checkpoint = results.experiment_analysis.best_checkpoint
    if checkpoint is None:
        raise RuntimeError("No hyperopt trial produced a checkpoint")
    with checkpoint.as_directory() as ckpt_path:
        model_dir = os.path.join(ckpt_path, MODEL_FILE_NAME)
        if not os.path.isdir(model_dir):
            raise RuntimeError(f"Best checkpoint has no model files at {model_dir}")
        return LudwigModel.load(model_dir, backend="local", from_checkpoint=True)


def train_ludwig(train_df, target_column, seed):
    """Runs plain Ludwig AutoML (auto_train) with its default configuration."""
    return auto_train(
        dataset=train_df,
        target=target_column,
        time_limit_s=LUDWIG_TIME_LIMIT,
        tune_for_memory=False,
        random_seed=seed,
        # Ludwig silently switches to its distributed Ray backend when
        # "ray" is importable; that backend fails in Ludwig 0.17.x with
        # "RayTrainerV2 has no attribute create_checkpoint_handle".
        # Hyperopt still uses Ray Tune; only each trial trains locally.
        backend="local",
    )


def extract_ludwig_diagnostics(results, model):
    """What Ludwig did during one run (trials, epochs, model, resources), for auditing.

    Answers questions such as "how many trials fitted in the time limit", "how
    long did the best one train" or "which model type was chosen". Every field
    is optional: a failure to read one never affects the metrics.
    """
    diag = {}
    try:
        analysis = results.experiment_analysis
        trials = list(analysis.trials)
        by_status = {}
        for t in trials:
            by_status[t.status] = by_status.get(t.status, 0) + 1
        best = analysis.best_trial
        last = best.last_result or {}
        diag.update({
            "n_trials": len(trials),
            "trials_by_status": by_status,
            "best_trial_id": best.trial_id,
            "best_trial_epochs": last.get("training_iteration"),
            "best_trial_time_s": last.get("time_total_s"),
            "sum_trial_time_s": round(sum(
                (t.last_result or {}).get("time_total_s", 0) for t in trials), 1),
            "best_hyperparameters": analysis.best_config,
        })
    except Exception as e:
        print(f"[WARN] Could not read Ludwig's trials: {e}")
    try:
        cfg = model.config
        trainer = cfg.get("trainer") or {}
        diag["model_type"] = cfg.get("model_type")
        diag["combiner"] = (cfg.get("combiner") or {}).get("type")
        diag["trainer"] = {k: trainer.get(k) for k in (
            "epochs", "batch_size", "learning_rate", "optimizer", "early_stop")}
    except Exception as e:
        print(f"[WARN] Could not read Ludwig's trainer settings: {e}")
    try:
        import ray
        if ray.is_initialized():  # still up here; cleanup_ludwig_artifacts() stops it later
            diag["ray_resources"] = ray.cluster_resources()
    except Exception as e:
        print(f"[WARN] Could not read Ray's resources: {e}")
    return diag


def run_ludwig_experiment(train_df, test_df, target_column, seed, task_type):
    """Trains and evaluates a single Ludwig AutoML run: (metrics, feature types, diagnostics)."""
    cleanup_ludwig_artifacts()  # stale hyperopt/ from a previous run
    try:
        train_start = time.perf_counter()
        results = train_ludwig(train_df, target_column, seed)
        train_time = time.perf_counter() - train_start
        model = load_best_model_local(results)
        eval_results, predictions, _ = model.evaluate(test_df, collect_predictions=True)
        metrics = extract_ludwig_metrics(eval_results, target_column, task_type)
        diagnostics = extract_ludwig_diagnostics(results, model)
        metrics["train_time_s"] = round(train_time, 2)  # auto_train only (time-limited)
        if "n_trials" in diagnostics:
            metrics["n_trials"] = diagnostics["n_trials"]
            metrics["n_trials_failed"] = diagnostics["trials_by_status"].get("ERROR", 0)
        if "classification" in task_type.lower():
            metrics.update(extra_classification_metrics(
                test_df[target_column], predictions[f"{target_column}_predictions"]))

        # Logs what Ludwig decided on its own (feature types, encoders, combiner), for
        # reproducibility and auditing. Read from the loaded best model's config: Ray's
        # best_config only holds the sampled hyperparameters, not the full Ludwig config.
        inferred_types = None
        try:
            cfg = model.config
            inferred_types = {
                "input_features": {
                    f["name"]: {"type": f["type"], "encoder": (f.get("encoder") or {}).get("type")}
                    for f in cfg["input_features"]
                },
                "output_features": {f["name"]: f["type"] for f in cfg["output_features"]},
                "combiner": (cfg.get("combiner") or {}).get("type"),
            }
        except Exception as e:
            print(f"[WARN] Could not extract Ludwig's inferred feature types: {e}")

        del results
        return metrics, inferred_types, diagnostics
    except Exception as e:
        print(f"[ERROR] Ludwig, seed {seed}: {e}")
        return None, None, None
    finally:
        # Metrics are already in memory; trial dumps are no longer needed
        cleanup_ludwig_artifacts()


def extract_ludwig_metrics(eval_results, target_column, task_type):
    metrics_dict = {}
    try:
        feature_metrics = eval_results.get(
            target_column, eval_results.get("combined", next(iter(eval_results.values())))
        )
        if "classification" in task_type.lower():
            for metric in ["loss", "roc_auc", "accuracy", "accuracy_micro",
                            "precision", "recall", "specificity"]:
                if metric in feature_metrics:
                    metrics_dict[metric] = safe_metric_value(feature_metrics[metric])
        else:
            for metric in ["loss", "root_mean_squared_error", "r2",
                            "mean_absolute_error", "mean_squared_error"]:
                if metric in feature_metrics:
                    metrics_dict[metric] = safe_metric_value(feature_metrics[metric])
    except Exception as e:
        print(f"[ERROR] Extracting Ludwig metrics: {e}")
    return metrics_dict

## Main loop

In [ ]:
script_dir = os.getcwd()
save_dir = os.path.join(script_dir, "results", RUN_MODE_NAME)
feature_type_log_path = os.path.join(save_dir, "ludwig_inferred_feature_types.json")
start_fresh_if_requested(save_dir)  # only acts with BENCH_RESUME=0
os.makedirs(save_dir, exist_ok=True)
save_environment_info(save_dir, "ludwig")
diagnostics_log_path = os.path.join(save_dir, "ludwig_run_diagnostics.json")


def load_json_log(path):
    """Previous log, to keep what already-finished runs recorded when resuming."""
    if RESUME and os.path.isfile(path):
        try:
            with open(path) as f:
                return json.load(f)
        except Exception as e:
            print(f"[WARN] Could not read {os.path.basename(path)}: {e}")
    return {}


def save_json_log(path, log):
    try:
        with open(path, "w") as f:
            json.dump(log, f, indent=2, default=str)
    except Exception as e:
        print(f"[WARN] Could not save {os.path.basename(path)}: {e}")


feature_type_log = load_json_log(feature_type_log_path)
diagnostics_log = load_json_log(diagnostics_log_path)

all_task_ids = TASK_IDS[:1] if TEST_MODE else TASK_IDS

for task_id in all_task_ids:
    print(f"\n=== Processing task {task_id} ===")
    try:
        task = openml.tasks.get_task(task_id)
    except Exception as e:
        print(f"[ERROR] Could not fetch task {task_id}: {e}")
        continue

    plan = get_fold_repeat_seed_plan(task)

    def forget_task_logs():
        """Drops what the discarded runs of this dataset recorded in the JSON logs."""
        for log in (feature_type_log, diagnostics_log):
            for key in [k for k in log if k.startswith(f"task_{task_id}_")]:
                del log[key]
        save_json_log(feature_type_log_path, feature_type_log)
        save_json_log(diagnostics_log_path, diagnostics_log)

    # A finished dataset is skipped; a half-done one is discarded and restarted from fold 0
    runs, (dataset_name, task_type, evaluation_measure) = load_completed_runs(
        save_dir, task_id, "ludwig", expected_runs=len(plan))
    if len(runs) < len(plan):
        forget_task_logs()

    for attempt in range(1 + DATASET_RETRIES):
        if len(runs) == len(plan):
            break
        if attempt > 0:
            print(f"[RETRY] task {task_id}: a run failed; discarding this dataset's runs "
                  f"and starting it over (attempt {attempt + 1} of {1 + DATASET_RETRIES})")
            archive_task_results(save_dir, task_id, "ludwig")
            forget_task_logs()
            runs = []
        for (repeat, fold, seed) in plan:
            split_data = download_openml_task_split(task, repeat, fold)
            if split_data is None:
                break

            dataset_name = split_data["dataset_name"]
            task_type = split_data["task_type"]
            evaluation_measure = split_data["evaluation_measure"]

            print(f"[RUN] task={task_id} repeat={repeat} fold={fold} seed={seed}")

            run_start = time.perf_counter()
            metrics, inferred_types, diagnostics = run_ludwig_experiment(
                split_data["train_df"], split_data["test_df"],
                split_data["target_column"], seed, task_type,
            )
            run_time = time.perf_counter() - run_start
            if not metrics:
                break  # a failed run: the whole dataset is retried
            metrics["time_taken"] = round(run_time, 2)  # seconds, train + eval
            runs.append({"repeat": repeat, "fold": fold, "seed": seed, "metrics": metrics})
            run_key = f"task_{task_id}_repeat{repeat}_fold{fold}_seed{seed}"
            if inferred_types:
                feature_type_log[run_key] = inferred_types
            if diagnostics:
                diagnostics_log[run_key] = diagnostics
            # Saved after every run so an interruption loses nothing
            save_json_log(feature_type_log_path, feature_type_log)
            save_json_log(diagnostics_log_path, diagnostics_log)
            save_incremental_results(
                task_id, "ludwig", runs, dataset_name, task_type, evaluation_measure, save_dir,
            )
    if len(runs) < len(plan):
        print(f"[WARN] task {task_id}: only {len(runs)} of {len(plan)} runs completed after "
              f"{1 + DATASET_RETRIES} attempt(s). Launch the benchmark again to restart it.")

    save_summary(save_dir, "ludwig")


print("\nAll tasks processed.")
print(f"Results in: {save_dir}")
print(f"Ludwig's inferred feature types: {feature_type_log_path}")
print(f"Ludwig's per-run diagnostics (trials, epochs, model): {diagnostics_log_path}")

## Analysis

Summary table and statistics from the results above (the same as `python analyze_results.py <mode>`). Edit `ANALYSIS_OUTPUTS` to choose which files are written.

In [ ]:
# Which analysis files to write; the summary table is always shown below.
# Available: summary_table, baseline, time_and_failures, ludwig_diagnostics,
# per_task_stats, paired_tests (the last needs both tools' results).
ANALYSIS_OUTPUTS = ["summary_table", "baseline", "time_and_failures", "ludwig_diagnostics", "per_task_stats", "paired_tests"]

try:
    from analyze_results import run_analysis  # lives in the parent folder
    from IPython.display import display

    summary = run_analysis(RUN_MODE_NAME, tool="ludwig", outputs=ANALYSIS_OUTPUTS,
                           time_limit=TIME_LIMIT_S)
    display(summary)
except (Exception, SystemExit) as e:
    print(f"[WARN] The analysis could not run ({e}). From a terminal: "
          f"python analyze_results.py {RUN_MODE_NAME} --tool ludwig")
